In [ ]:
# -----------------------------
# Imports
# -----------------------------
import pandas as pd
import numpy as np
import torch
import os
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# -----------------------------
# Load ESCOTaxonomy Dataset
# -----------------------------
ds = load_dataset("frtna/ESCOTaxonomy")
df = pd.DataFrame(ds['train'])
# Allow commas with optional spaces
df['skills_list'] = df['skills'].str.split(r',\s*')

print(df.iloc[0])

In [ ]:
# -----------------------------
# Data exploration to understand skill distribution
# -----------------------------
all_skills_temp = []
for skills in df['skills_list']:
    all_skills_temp.extend(skills)
unique_skills_temp = list(set(all_skills_temp))
skill_counts = pd.Series(all_skills_temp).value_counts()

print("=== Dataset Overview ===")
print(f"Total jobs: {len(df)}")
print(f"Unique skills: {len(unique_skills_temp)}")
print(f"\nSkills per job - Mean: {df['skills_list'].apply(len).mean():.1f}, Median: {df['skills_list'].apply(len).median():.0f}")
print(f"Rare skills (appear once): {(skill_counts == 1).sum()} ({(skill_counts == 1).sum()/len(skill_counts)*100:.1f}%)")

In [ ]:
# -----------------------------
# Check skill distribution uniformity with visualization
# -----------------------------
all_skills = []
for skills in df['skills_list']:
    all_skills.extend(skills)

skill_counts = pd.Series(all_skills).value_counts()

# Create visualization

plt.figure(figsize=(8, 5))
plt.hist(skill_counts.values, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
plt.axvline(skill_counts.mean(), color='red', linestyle='--', label=f'Mean: {skill_counts.mean():.1f}')
plt.axvline(skill_counts.mean() + skill_counts.std(), color='orange', linestyle='--', label=f'Mean+Std: {skill_counts.mean() + skill_counts.std():.1f}')
plt.xlabel('Skill Frequency')
plt.ylabel('Number of Skills')
plt.title('Skill Frequency Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Simple statistical check
print(f"Mean: {skill_counts.mean():.2f}, Std: {skill_counts.std():.2f}")
print(f"Std/Mean ratio: {skill_counts.std() / skill_counts.mean():.2f}")

# Heuristic approach: If std is close to or less than mean, distribution is more uniform; if std is much greater than mean, it's skewed
if skill_counts.std() <= skill_counts.mean():
    print("Skills distribute uniformly")
else:
    print("Skills are skewed")


In [ ]:
# -----------------------------
# Prepare training pairs for semantic model
# -----------------------------

# We'll use job description + title as input
# Target: skills (you can also include synonyms as positive text)

# Create training pairs: (job_description, skill)
train_examples = []
for _, row in df.iterrows():
    for skill in row['skills_list']:
        train_examples.append(InputExample(texts=[row['description'], skill]))


In [ ]:
# -----------------------------
# Split into train/test sets
# -----------------------------
# Using stratified sampling based on skills, with random sampling for rare skills with <= 20 occurrences

# Create labels for stratification based on the skill in each example
stratify_labels = [example.texts[1] for example in train_examples]

# Count skill occurrences
skill_counts = Counter(stratify_labels)

min_samples_for_stratification = 20

# Separate examples based on skill frequency
stratifiable_examples = []
stratifiable_labels = []
non_stratifiable_examples = []

for ex, skill in zip(train_examples, stratify_labels):
    if skill_counts[skill] >= min_samples_for_stratification:
        stratifiable_examples.append(ex)
        stratifiable_labels.append(skill)
    else:
        non_stratifiable_examples.append(ex)

print(f"Stratifiable examples: {len(stratifiable_examples)} ({len(set(stratifiable_labels))} unique skills)")
print(f"Non-stratifiable examples: {len(non_stratifiable_examples)} (rare skills)")

# Perform stratified split on frequent skills
if len(stratifiable_examples) > 0:
    train_ex_stratified, val_ex_stratified = train_test_split(
        stratifiable_examples, 
        test_size=0.1, 
        random_state=42,
        stratify=stratifiable_labels
    )
else:
    train_ex_stratified = []
    val_ex_stratified = []

# Random split for non-stratifiable examples (rare skills)
if len(non_stratifiable_examples) > 0:
    train_ex_random, val_ex_random = train_test_split(
        non_stratifiable_examples,
        test_size=0.1,
        random_state=42
    )
else:
    train_ex_random = []
    val_ex_random = []

# Combine both splits
train_ex = train_ex_stratified + train_ex_random
val_ex = val_ex_stratified + val_ex_random

print(f"\nFinal split:")
print(f"Training pairs: {len(train_ex)} (stratified: {len(train_ex_stratified)}, random: {len(train_ex_random)})")
print(f"Validation pairs: {len(val_ex)} (stratified: {len(val_ex_stratified)}, random: {len(val_ex_random)})")
denom = len(train_ex) + len(val_ex)
if denom > 0:
    print(f"Split ratio: {len(val_ex)/denom*100:.1f}% validation")
else:
    print("Split ratio: N/A (no examples)")

In [ ]:
# -----------------------------
# Fine-tune SentenceTransformer
# -----------------------------

# MultipleNegativesRankingLoss treats other skills in batch as negatives
# This learns to map similar job descriptions and skills close together
model_dir = './skill_model'
# If a fine-tuned model already exists, load it and skip training
if os.path.isdir(model_dir) and len(os.listdir(model_dir)) > 0:
    print(f'Found existing model at {model_dir}, loading...')
    model = SentenceTransformer(model_dir)
else:
    print('No saved model found — initializing and (possibly) fine-tuning...')
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    if len(train_ex) == 0:
        print('No training examples available — skipping training')
    else:
        train_dataloader = DataLoader(train_ex, shuffle=True, batch_size=16)
        train_loss = losses.MultipleNegativesRankingLoss(model=model)
        model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=1,
            warmup_steps=100,
            output_path=model_dir,
        )
        print('Model fine-tuned and saved!')
        # Reload saved model to ensure consistent state
        model = SentenceTransformer(model_dir)

In [ ]:
# -----------------------------
# Build ESCO embeddings index
# -----------------------------

# Build skill embeddings index for inference
# Encode all unique skills once, then use cosine similarity for retrieval

# Create embeddings for all unique skills
all_skills = []
for skills in df['skills_list']:
    all_skills.extend(skills)
# Use deterministic ordering so indices are stable across runs
unique_skills = sorted(set(all_skills))

print(f"Encoding {len(unique_skills)} unique skills...")
skill_embeddings = model.encode(unique_skills, show_progress_bar=True)

# Test on a sample job description
test_job = df.sample(n=1, random_state=42).iloc[0]
print(f"\nTest job: {test_job['job_title']}")
print(f"Description: {test_job['description'][:200]}...")
print(f"\nTrue skills: {test_job['skills'][:200]}...")

# Predict skills
job_embedding = model.encode([test_job['description']])
similarities = cosine_similarity(job_embedding, skill_embeddings)[0]

# Get top 10 predictions
top_k = 10
top_indices = np.argsort(similarities)[-top_k:][::-1]

print(f"\nTop {top_k} predicted skills (with confidence):")
for idx in top_indices:
    print(f"  {similarities[idx]:.3f} - {unique_skills[idx]}")


## Calibration Analysis

The model produces similarity-based confidence scores for retrieved skills rather than calibrated probabilistic predictions over the full label space. Therefore, Brier Score is not applicable, as it requires well-defined probabilities for each possible class. However, Expected Calibration Error (ECE) can be computed by treating cosine similarity scores as proxy confidence values to assess how well similarity correlates with empirical correctness, although this does not represent true probabilistic calibration. So we compute calibration of similarity scores, NOT true probabilistic calibration.

In [ ]:
# -----------------------------
# Evaluate performance
# -----------------------------

# Systematic evaluation on held-out test set
# Metric: Precision@5
# Includes bootstrap confidence intervals and calibration analysis

# Prepare test set
test_df = df.sample(n=300, random_state=42)

def predict_top_k_skills(description, k=5):
    job_emb = model.encode([description])
    sims = cosine_similarity(job_emb, skill_embeddings)[0]
    top_indices = np.argsort(sims)[-k:][::-1]
    return [(unique_skills[idx], sims[idx]) for idx in top_indices]

def evaluate_model(test_df, k=5):
    precisions = []
    all_confidences = []
    all_correct = []
    
    for _, row in test_df.iterrows():
        predictions = predict_top_k_skills(row['description'], k=k)
        true_skills = set(row['skills_list'])
        
        hits = 0
        for skill, conf in predictions:
            all_confidences.append(conf)
            is_correct = 1 if skill in true_skills else 0
            all_correct.append(is_correct)
            hits += is_correct
        
        precisions.append(hits / k)
    
    return np.array(precisions), np.array(all_confidences), np.array(all_correct)

# Evaluate
precisions, confidences, correct = evaluate_model(test_df, k=5)
mean_precision = np.mean(precisions)
print(f"Precision@5: {mean_precision:.4f}")

# Bootstrap 95% CI for Precision@k 
np.random.seed(42)
bootstrap_means = []
bootstrap_iterations = 1000
for _ in range(bootstrap_iterations):
    sample_idx = np.random.choice(len(precisions), size=len(precisions), replace=True)
    bootstrap_means.append(np.mean(precisions[sample_idx]))

ci_lower, ci_upper = np.percentile(bootstrap_means, [2.5, 97.5])
print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

# Expected Calibration Error
def calculate_ece(confidences, correct, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        # Make last bin inclusive of 1.0 to avoid excluding perfect-confidence predictions
        if i < n_bins - 1:
            mask = (confidences >= bin_boundaries[i]) & (confidences < bin_boundaries[i+1])
        else:
            mask = (confidences >= bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        if np.sum(mask) > 0:
            bin_conf = np.mean(confidences[mask])
            bin_acc = np.mean(correct[mask])
            ece += np.sum(mask) / len(confidences) * np.abs(bin_acc - bin_conf)
    return ece

ece = calculate_ece(confidences, correct)
print(f"ECE: {ece:.4f}")
